#Cell 1 — Check versions (already done, skip if done)

In [4]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes huggingface_hub sentencepiece protobuf

In [8]:
import transformers, trl, peft, accelerate, bitsandbytes
print("transformers:", transformers.__version__)
print("trl         :", trl.__version__)
print("peft        :", peft.__version__)
print("accelerate  :", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

transformers: 4.51.3
trl         : 0.19.1
peft        : 0.19.1
accelerate  : 1.13.0
bitsandbytes: 0.49.2


#Cell 2 — Imports

In [9]:
import os
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig

print("All imports OK ✓")

All imports OK ✓


In [ ]:
#Cell 3 — Config

In [10]:
MODEL_ID     = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR   = "/content/qwen-bengali"
MAX_SEQ_LEN  = 1024
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
EPOCHS       = 3
LR           = 2e-4
WARMUP_RATIO = 0.05
SAVE_STEPS   = 100
LOG_STEPS    = 10

#Cell 4 — BnB Config

In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

#Cell 5 — Tokenizer

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {tokenizer.vocab_size}")
print(f"Pad token  : {tokenizer.pad_token!r}")
print("Tokenizer loaded ✓")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size : 151643
Pad token  : '<|endoftext|>'
Tokenizer loaded ✓


#Cell 6 — Load Model

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("Model loaded ✓")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

...

Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded ✓


#Cell 7 — LoRA

In [14]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 161,480,704 || all params: 7,777,097,216 || trainable%: 2.0764


In [15]:
ds1 = load_dataset("json", data_files="/content/train.jsonl",    split="train")
ds2 = load_dataset("json", data_files="/content/gk_train.jsonl", split="train")

dataset = concatenate_datasets([ds1, ds2])
print(f"Total examples : {len(dataset)}")
print(f"Columns        : {dataset.column_names}")

split         = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train : {len(train_dataset)}")
print(f"Eval  : {len(eval_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total examples : 2372
Columns        : ['text']
Train : 2253
Eval  : 119


#Cell 8 — Load & Merge Datasets

In [16]:
print("Sample record:")
print(train_dataset[0]['text'][:300])

Sample record:
<start_of_turn>user
বাংলাদেশের সম্পর্কিত এই তথ্যটি কী?<end_of_turn>
<start_of_turn>model
• বাংলাদেশের সর্বোচ্চ পর্বতশৃঙ্গ হলো তাজিংডং, যা বিজয় নামেও পরিচিত।<end_of_turn>


#Cell 9 — Check dataset format

In [17]:
def verify_format(example):
    text = example["text"]
    assert "user" in text.lower(),      "Missing user turn"
    assert "assistant" in text.lower() or "model" in text.lower(), "Missing assistant turn"

verify_format(train_dataset[0])
print("Format check passed ✓")

Format check passed ✓


#Cell 11 — SFT Config

In [18]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    logging_steps=LOG_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataset_text_field="text",
)

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


#Cell 12 — Trainer

In [20]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,      # ← changed from tokenizer=
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print("Trainer ready ✓")

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:412: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:458: UserWarning: You are using packing, but the attention implementation is not set to 'flash_attention_2'. Packing flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to cross-contamination between batches. To avoid this, ei

Adding EOS to train dataset:   0%|          | 0/2253 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2253 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (225630 > 131072). Running this sequence through the model will result in indexing errors


Packing train dataset:   0%|          | 0/2253 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Trainer ready ✓


#Cell 13 — Train 🚀

In [21]:
print("Starting training...")
print(f"  Model          : {MODEL_ID}")
print(f"  Train examples : {len(train_dataset)}")
print(f"  Epochs         : {EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Max seq len    : {MAX_SEQ_LEN}")
print()
trainer.train()

Starting training...
  Model          : Qwen/Qwen2.5-7B-Instruct
  Train examples : 2253
  Epochs         : 3
  Effective batch: 16
  Max seq len    : 1024



Step,Training Loss,Validation Loss


TrainOutput(global_step=72, training_loss=0.3179163994888465, metrics={'train_runtime': 508.2798, 'train_samples_per_second': 2.302, 'train_steps_per_second': 0.142, 'total_flos': 4.917830784039936e+16, 'train_loss': 0.3179163994888465})

#Cell 14 — Save Adapter

In [22]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved → {ADAPTER_DIR}")

Adapter saved → /content/qwen-bengali/lora_adapter


#Cell 15 — Test Inference

In [23]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
inf_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
inf_model.eval()

def test_inference(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(inf_model.device)
    with torch.no_grad():
        outputs = inf_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    print(f"Prompt   : {prompt}")
    print(f"Response : {response}\n")

test_inference("বিরোধ যোজক কী? উদাহরণসহ ব্যাখ্যা করো।")
test_inference("১৯৫৫ সালে বাংলাদেশে কোন গ্যাসক্ষেত্র আবিষ্কৃত হয়?")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Prompt   : বিরোধ যোজক কী? উদাহরণসহ ব্যাখ্যা করো।
Response : • বিরোধ যোজক একটি ভৌগোলিক ওয়েটলান্ড এলাকা, যা সমুদ্র জলের সঙ্গে প্রায় একই তুলনামূলক অবস্থান করে থাকে এবং একটি প্রত্নতাত্ত্বিক শালভূমি হিসেবে পরিচিত।
• বড়পুকুরিয়া ইউনিয়নের বিরোধ যোজক দিয়ার দৈর্ঘ্য ১৩৫ কিলোমিটার এবং প্রস্থ ১০

Prompt   : ১৯৫৫ সালে বাংলাদেশে কোন গ্যাসক্ষেত্র আবিষ্কৃত হয়?
Response : • বাংলাদেশে প্রথম বার তেলক্ষেত্র আবিষ্কৃত হয় ১৯৫৫ সালে চট্টগ্রামের হরিপুরে।

